In [2]:
import pandas as pd
import numpy as np

def kaufman_efficiency_ratio(close_series, period=10):
    # Variation nette sur la période (entre maintenant et il y a N périodes)
    net_change = np.abs(close_series - close_series.shift(period))
    
    # Somme des variations absolues bar à bar sur la période
    change = np.abs(close_series - close_series.shift(1))
    volatility = change.rolling(window=period).sum()
    
    # Éviter la division par zéro
    er = np.where(volatility == 0, 0, net_change / volatility)
    
    return pd.Series(er, index=close_series.index)

# Exemple d'utilisation :
# df['KER'] = kaufman_efficiency_ratio(df['Close'], period=10)

In [3]:
from ta.volatility import AverageTrueRange

df = pd.read_csv("./Trading Data/GBPJPY_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df = df.loc['2022-05-01' : '2026-9-16']

indicator_er = kaufman_efficiency_ratio(df['Close'], 10)
print(indicator_er.values)

for row in indicator_er.values:
    print(row)
    
# indicator_atr_20 = AverageTrueRange(df['High'], df['Low'], df['Close'], window=20)
# indicator_atr_100 = AverageTrueRange(df['High'], df['Low'], df['Close'], window=100)
# df['ATRCOMPRESS'] = indicator_atr_20._atr /indicator_atr_100._atr


[       nan        nan        nan ... 0.24880055 0.14106583 0.02608696]
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
0.17969379532634366
0.1956521739130563
0.11904761904760952
0.24593716143010724
0.34769230769231846
0.32698412698413654
0.5694618272841566
0.6993166287016334
0.6708229426434352
0.9458937198067778
0.9518486672399106
0.9546191247974197
0.49962769918092587
0.5073313782991133
0.4610038610038399
0.3703148425787017
0.4060606060605917
0.31651749389747497
0.2650793650793576
0.0854870775347708
0.43661971830983576
0.6250000000000218
0.46031746031746984
0.5054545454545458
0.709401709401712
0.5865070729053291
0.29946524064173397
0.07641633728589431
0.06575185820469484
0.16462167689162027
0.03402646502834394
0.1151515151515235
0.07154742096505627
0.10862619808306281
0.3014533520862624
0.2680608365018961
0.18518518518519297
0.049046321525886824
0.15276630883567327
0.5108225108225163
0.27575757575759013
0.16870415647926212
0.4880952380953076
0.23555555555557914
0.5360000000000055
0.691629955

In [4]:
import numpy as np
import pandas as pd


def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = "NEUTRAL"

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [5]:
import pandas as pd
import time

from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

global_range_to_consider = 0
global_tp_multiple = 0
global_stop_trade_hour = 0
global_close_range_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 3
    tp_multiple = 5.5
    stop_trade_hour = 18
    close_range_hour = 8
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.opening_range_high = None
        self.opening_range_low = None
        if global_range_to_consider != 0:
            self.range_to_consider = global_range_to_consider

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_close_range_hour != 0:
            self.close_range_hour = global_close_range_hour

        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        self.current_day = day
        self.current_day_position_taked = False
        self.opening_range_high = None
        self.opening_range_low = None

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price
        print(f"share_by_risk = {share_by_risk} & share_by_leverage = {share_by_leverage}")

        return int(min(share_by_risk, share_by_leverage))

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()

        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
            # print(f"new date {current_bar_date}")

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time().hour < self.close_range_hour:
                return

        if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high and self.data['market_state'] == 'BULLISH'
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low and self.data['market_state'] == 'BEARISH'
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = t.time().hour < self.stop_trade_hour
            c5 = self.data['ER'][-1] > self.er15_threshold

            if not self.current_day_position_taked:
                if c_long and c2 and c3 and c4 and c5:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_low
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price + per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.equity) / per_share_risk)

                    # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True
                if c_short and c2 and c3 and c4 and c5:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price - per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True

In [6]:
from datetime import datetime

from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

window = 2
global_range_to_consider = 3
global_close_range_hour = 7
global_stop_trade_hour = 18
global_tp_multiple = 5
global_er15_threshold = 0.3

# start_date = '2021-3-1 00:00'
# end_date = '2023-3-30 00:00'
start_date = '2020-1-1 00:00'
end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)

df_result = implement_market_structure_states_strict(df_d1, n=window)
df_result['ER'] = indicator_er.values

# df['market_state'] = df_result["market_state"]

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    # er is previous day er
    if i == 0:
        er = row['ER']
    else :
        er = df_result.iloc[i-1]['ER']

    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(row['market_state'])
        h1_tf_er15_list.append(er)

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

# plt.plot(stats._equity_curve.index, stats._equity_curve['Equity'])

bt.plot()


Start                     2020-01-01 22:00...
End                       2026-09-16 01:00...
Duration                   2449 days 02:00:00
Exposure Time [%]                    17.89164
Equity Final [$]                    24143.238
Equity Peak [$]                     25162.705
Return [%]                          141.43238
Buy & Hold Return [%]                45.36014
Return (Ann.) [%]                    11.17347
Volatility (Ann.) [%]                16.35659
CAGR [%]                             14.04839
Sharpe Ratio                          0.68312
Sortino Ratio                         1.46833
Calmar Ratio                          0.52852
Alpha [%]                           134.02175
Beta                                  0.16337
Max. Drawdown [%]                   -21.14112
Avg. Drawdown [%]                    -1.89084
Max. Drawdown Duration      994 days 13:00:00
Avg. Drawdown Duration       22 days 22:00:00
# Trades                                  166
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1261', ...)

In [19]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

window = 2
global_range_to_consider = 3
global_close_range_hour = 7
global_stop_trade_hour = 18
global_tp_multiple = 5

start_date = '2020-1-1 00:00'
end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

df_result = implement_market_structure_states_strict(df_d1, n=window)

# df['market_state'] = df_result["market_state"]

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

er15_window_list = [10,15,20,25]
er15_threshold_list = [0.3,0.4,0.5,0.6]

with open(f"./results/result_{pair}_{time.time()}.csv", "a") as f:
    f.write(f"Window,RTC,CRH,STH,TpMultiple,Return,ReturnYearly,MDrawdown,WinRate,NumTrades,ER15Window,ER15Threshold\n")

    for er15_window in er15_window_list:
        for er15_threshold in er15_threshold_list:
            global_er15_threshold = er15_threshold

            df_h1_test = df_h1.copy()
            indicator_er = kaufman_efficiency_ratio(df_d1['Close'], er15_window)
            df_result['ER'] = indicator_er.values

            h1_tf_direction_list = []
            h1_tf_er15_list = []

            for i in range(len(df_result)):
                row = df_result.iloc[i]
                # er is previous day er
                if i == 0:
                    er = row['ER']
                else :
                    er = df_result.iloc[i-1]['ER']

                df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
                for index2, row2 in df_by_day.iterrows():
                    h1_tf_direction_list.append(row['market_state'])
                    h1_tf_er15_list.append(er)
            
            df_h1_test['market_state'] = h1_tf_direction_list
            df_h1_test['ER'] = h1_tf_er15_list
            df_h1_test.index = pd.to_datetime(df_h1_test.index, utc=True).tz_convert(timezone)

            bt = Backtest(df_h1_test, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
            stats = bt.run()

            f.write(f"{window},{global_range_to_consider},{global_close_range_hour},{global_stop_trade_hour},{global_tp_multiple},{stats['Return [%]']},{stats['Return (Ann.) [%]']},{stats['Max. Drawdown [%]']},{stats['Win Rate [%]']}, {stats['# Trades']}, {er15_window}, {er15_threshold}\n")

            print(f"d1 window = {window} & range to consider = {global_range_to_consider} & close range hour = {global_close_range_hour} & stop_trade_hour = {global_stop_trade_hour} & tp_multiple = {global_tp_multiple} & er15_window = {er15_window} & er15_threshold = {er15_threshold} DONE")
        
        

# plt.plot(stats._equity_curve.index, stats._equity_curve['DrawdownPct'])


d1 window = 2 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 & er15_window = 10 & er15_threshold = 0.3 DONE
d1 window = 2 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 & er15_window = 10 & er15_threshold = 0.4 DONE
d1 window = 2 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 & er15_window = 10 & er15_threshold = 0.5 DONE
d1 window = 2 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 & er15_window = 10 & er15_threshold = 0.6 DONE
d1 window = 2 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 & er15_window = 15 & er15_threshold = 0.3 DONE
d1 window = 2 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 & er15_window = 15 & er15_threshold = 0.4 DONE
d1 window = 2 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 5 & er

In [23]:
import pandas as pd
import numpy as np
import time

from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

global_er15_threshold = 0.3

start_date = '2020-01-01 00:00'
end_date = '2026-9-17 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

d1_window_list = [1,2,4]
range_to_consider_list = [3,4]
close_range_hour_list = [7,8]
stop_trade_hour_list = [18,19,21]
tp_multiple_list = [3,4,5,6]

with open(f"./results/result_{pair}_{time.time()}.csv", "a") as f:
    f.write(f"Window,RTC,CRH,STH,TpMultiple,Return,ReturnYearly,MDrawdown,WinRate,NumTrades\n")

    for window in d1_window_list:
        df_h1_test = df_h1.copy()
        df_result = implement_market_structure_states_strict(df_d1, n=window)

        indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
        df_result['ER'] = indicator_er.values

        h1_tf_direction_list = []
        h1_tf_er15_list = []

        for i in range(len(df_result)):
            row = df_result.iloc[i]
            # er is previous day er
            if i == 0:
                er = row['ER']
            else :
                er = df_result.iloc[i-1]['ER']

            df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
            for index2, row2 in df_by_day.iterrows():
                h1_tf_direction_list.append(row['market_state'])
                h1_tf_er15_list.append(er)
        
        df_h1_test['market_state'] = h1_tf_direction_list
        df_h1_test['ER'] = h1_tf_er15_list
        df_h1_test.index = pd.to_datetime(df_h1_test.index, utc=True).tz_convert(timezone)

        for range_to_consider in range_to_consider_list:
            for close_range_hour in close_range_hour_list:
                for stop_trade_hour in stop_trade_hour_list:
                    for tp_multiple in tp_multiple_list:
                        # range_to_consider, tp_multiple, stop_trade_hour, close_range_hour
                        global_range_to_consider = range_to_consider
                        global_close_range_hour = close_range_hour
                        global_stop_trade_hour = stop_trade_hour
                        global_tp_multiple = tp_multiple

                        bt = Backtest(df_h1_test, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                        stats = bt.run()

                        f.write(f"{window},{range_to_consider},{close_range_hour},{stop_trade_hour},{tp_multiple},{stats['Return [%]']},{stats['Return (Ann.) [%]']},{stats['Max. Drawdown [%]']},{stats['Win Rate [%]']}, {stats['# Trades']}\n")

                        print(f"d1 window = {window} & range to consider = {range_to_consider} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple} DONE")



d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 3 DONE
d1 window = 1 & range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 4 DONE


KeyboardInterrupt: 

In [67]:
from datetime import datetime

from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

window = 2
global_range_to_consider = 3
global_close_range_hour = 7
global_stop_trade_hour = 18
global_tp_multiple = 5
global_er15_threshold = 0.3


start_date = '2020-1-1 00:00'
end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)

df_result = implement_market_structure_states_strict(df_d1, n=window)
df_result['ER'] = indicator_er.values

# df['market_state'] = df_result["market_state"]

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    # er is previous day er
    if i == 0:
        er = row['ER']
    else :
        er = df_result.iloc[i-1]['ER']

    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(row['market_state'])
        h1_tf_er15_list.append(er)

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

print (stats._equity_curve)

# print (stats._equity_curve.index.date)
# print (stats._equity_curve['Equity'])
# print (stats._equity_curve['Equity'].index)
# print (stats._equity_curve['Equity'].groupby([stats._equity_curve['Equity'].index.date])())

current_date = None
first_hour = None
last_hour = None
daily_equity = 0
daily_list = []
daily_equity_list = []
for index in range(len(stats._equity_curve)):
    row = stats._equity_curve.iloc[index]
    if current_date is None:
        current_date = row.name
        first_hour = row.Equity
    else:
        if current_date.date() != row.name.date():
            daily_list.append(current_date)
            daily_equity_list.append(stats._equity_curve.iloc[index-1].Equity)
            current_date = None

print(len(daily_list))         
print(len(daily_equity_list))         

with open(f"./results/TREND_DAILY_EQUITY_{pair}.csv", "a") as f:
    f.write(f"Date,Equity\n")

    for i in range(len(daily_list)):
        f.write(f"{daily_list[i]},{daily_equity_list[i]}\n")


Start                     2020-01-01 22:00...
End                       2026-09-16 01:00...
Duration                   2449 days 02:00:00
Exposure Time [%]                    17.89164
Equity Final [$]                    24143.238
Equity Peak [$]                     25162.705
Return [%]                          141.43238
Buy & Hold Return [%]                45.36014
Return (Ann.) [%]                    11.17347
Volatility (Ann.) [%]                16.35659
CAGR [%]                             14.04839
Sharpe Ratio                          0.68312
Sortino Ratio                         1.46833
Calmar Ratio                          0.52852
Alpha [%]                           134.02175
Beta                                  0.16337
Max. Drawdown [%]                   -21.14112
Avg. Drawdown [%]                    -1.89084
Max. Drawdown Duration      994 days 13:00:00
Avg. Drawdown Duration       22 days 22:00:00
# Trades                                  166
Win Rate [%]                      